# Đánh giá final toàn hệ thống trên Kaggle

Chạy **hệ thống thật đầy đủ** (PostgreSQL 16 + pgvector, Redis, Neo4j 5.18, Ollama: LLM #1 `qwen2.5:7b-instruct`,
LLM #2 `qwen-huhu`, giám khảo `gemma3:4b`) và 3 baseline trên bộ test.

**Cài đặt notebook:** Accelerator = **GPU T4 x2**, Internet = **On**.

**Input (Add Input):**
1. Dataset `huhu-system-final` (file `huhu-system-final.zip` tạo bằng `scripts/make_kaggle_package.py`).
2. File `qwen2.5-7b-huhu-Q4_K_M.gguf`: dataset riêng, hoặc Output của notebook `export_gguf_kaggle`.
3. (Chỉ khi chạy tiếp) Output của lần chạy trước của notebook này: các ca đã xong được bỏ qua.

Thứ tự: chép code -> kiểm tra giống hệt bản local (sha256 từng file, GGUF, digest model) -> dựng hạ tầng -> seed DB ->
kiểm DB -> unit test -> chạy thử -> crisis, retrieval -> P, B1, B2, B3 -> chấm -> bảng tổng hợp -> nén kết quả.
Bước kiểm tra nào lỗi thì notebook dừng (raise), không chạy test.

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time
W = "/kaggle/working"
APP = f"{W}/AIM/Hybrid-database-main"

def find(pattern):
    return sorted(p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

def sh(cmd, log=None):
    """Chạy lệnh shell trong thư mục app, in log trực tiếp; lỗi thì dừng notebook."""
    print("$", cmd, flush=True)
    full = "set -o pipefail; " + cmd + (f" 2>&1 | tee -a '{log}'" if log else "")
    r = subprocess.run(["bash", "-c", full], cwd=APP if os.path.isdir(APP) else W)
    if r.returncode:
        raise RuntimeError(f"Lệnh lỗi ({r.returncode}): {cmd}")

# 1. Chép code + bộ test vào /kaggle/working (Kaggle tự giải nén zip khi tạo dataset).
#    Luôn chép lại từ dataset (bản mới nhất); chỉ giữ kết quả đã chạy trong logs/ của lần trước cùng session.
keep = f"{W}/_logs_keep"
if os.path.isdir(f"{APP}/logs"):
    shutil.rmtree(keep, ignore_errors=True)
    shutil.move(f"{APP}/logs", keep)
shutil.rmtree(f"{W}/AIM", ignore_errors=True)
src = find("AIM/manifest.json")
if src:
    shutil.copytree(os.path.dirname(src[0]), f"{W}/AIM")
else:
    zips = find("huhu-system-final.zip")
    assert zips, "Không thấy dataset huhu-system-final"
    shutil.unpack_archive(zips[0], W)
MANIFEST = json.load(open(f"{W}/AIM/manifest.json", encoding="utf-8"))
print("manifest:", MANIFEST["created"], len(MANIFEST["files"]), "file")

In [ ]:
# 2. Kiểm tra liêm chính: mọi file code / dữ liệu / bộ test giống hệt bản local (trước khi tạo thêm file nào)
bad = [arc for arc, h in MANIFEST["files"].items()
       if not os.path.exists(os.path.join(W, arc)) or hashlib.sha256(open(os.path.join(W, arc), "rb").read()).hexdigest() != h]
if bad:
    raise RuntimeError(f"{len(bad)} file khác bản local: {bad[:10]}")
print(f"OK: {len(MANIFEST['files'])} file khớp sha256 với bản local")
shutil.copyfile(f"{APP}/dot_env", f"{APP}/.env")  # .env được đóng gói dưới tên dot_env

# 3. Chạy tiếp: lấy kết quả đã có (cùng session, hoặc Output của lần chạy trước gắn làm input)
if os.path.isdir(keep):
    shutil.copytree(keep, f"{APP}/logs", dirs_exist_ok=True)
    print("Giữ kết quả cũ trong session:", os.listdir(f"{APP}/logs"))
prev = find("AIM/Hybrid-database-main/logs/eval")
if prev:
    shutil.copytree(prev[0], f"{APP}/logs/eval", dirs_exist_ok=True)
    print("Chạy tiếp từ:", prev[0])
os.makedirs(f"{APP}/logs/eval", exist_ok=True)

In [ ]:
# 4. Hạ tầng (thay docker-compose): PostgreSQL 16 + pgvector, Redis, Neo4j 5.18, Ollama cùng bản local
# Bỏ ký tự CR (xuống dòng Windows) nếu có, rồi mới chạy: bash báo "invalid option name" với file CRLF
ks = f"{APP}/serving/kaggle_services.sh"
open(ks, "wb").write(open(ks, "rb").read().replace(b"\r\n", b"\n"))
sh("bash serving/kaggle_services.sh", log=f"{W}/setup.log")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# 5. Thư viện Python (torch có sẵn trên Kaggle)
sh("pip install -q 'psycopg2-binary>=2.9.9' 'pgvector>=0.2.5' 'neo4j>=5.19.0' 'redis>=5.0.3' "
   "'sentence-transformers>=2.6.1' 'python-dotenv>=1.0.0' 'httpx>=0.27.0' 'fastapi>=0.110.0' 'pydantic>=2.6.0'")

In [ ]:
# 6. Model: GGUF fine-tune khớp sha256 bản local; digest 3 model Ollama khớp `ollama list` ở local
ggufs = find("qwen2.5-7b-huhu-Q4_K_M.gguf")
assert ggufs, "Chưa gắn input chứa qwen2.5-7b-huhu-Q4_K_M.gguf"
h = hashlib.sha256()
with open(ggufs[0], "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == MANIFEST["gguf_sha256"], f"GGUF khác bản local: {h.hexdigest()}"
print("OK: GGUF khớp sha256")
link = f"{APP}/serving/qwen2.5-7b-huhu-Q4_K_M.gguf"
if not os.path.exists(link):
    os.symlink(ggufs[0], link)
sh("cd serving && ollama create qwen-huhu -f Modelfile")
sh("ollama pull qwen2.5:7b-instruct && ollama pull gemma3:4b")
out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(out)
have = {l.split()[0]: l.split()[1] for l in out.splitlines()[1:] if l.strip()}
diff = {m: (have.get(m), d) for m, d in MANIFEST["ollama_digests"].items() if have.get(m) != d}
if diff:
    raise RuntimeError(f"Model khác bản local (Kaggle, local): {diff}")
print("OK: 3 model khớp digest bản local")

In [ ]:
# 7. Embedding bge-m3 đúng revision đã dùng ở local. Giữ nguyên .env của hệ thống (đã kiểm sha256),
#    chỉ trỏ EMBEDDING_MODEL_NAME tới bản đã tải bằng biến môi trường (backend/ingestion đọc os.getenv).
from huggingface_hub import hf_hub_download, snapshot_download
B = MANIFEST["bge_m3"]
# Giống local: cấu hình + tokenizer từ main, trọng số model.safetensors từ nhánh chuyển đổi (main không có safetensors)
bge = snapshot_download("BAAI/bge-m3", revision=B["revision"],
                        allow_patterns=["*.json", "*.md", "1_Pooling/*", "sentencepiece.bpe.model"])
st = hf_hub_download("BAAI/bge-m3", "model.safetensors", revision=B["safetensors_revision"])
h = hashlib.sha256()
with open(st, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == B["safetensors_sha256"], f"bge-m3 model.safetensors khác bản local: {h.hexdigest()}"
if not os.path.exists(f"{bge}/model.safetensors"):
    os.symlink(os.path.realpath(st), f"{bge}/model.safetensors")
print("OK: bge-m3 khớp bản local:", sorted(os.listdir(bge)))
os.environ["EMBEDDING_MODEL_NAME"] = bge
os.environ["HF_HUB_OFFLINE"] = "1"
# bge-m3 chạy trên CPU (Ollama đã khởi động ở cell 4 nên vẫn dùng 2 GPU): lần chạy grounding trước, mỗi luồng nạp
# bge-m3 lên GPU làm Ollama phải đẩy một phần qwen-huhu sang CPU (6/29 lớp trên GPU) và chạy chậm nhiều lần.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
env = dict(l.split("=", 1) for l in open(f"{APP}/.env", encoding="utf-8").read().splitlines() if "=" in l and not l.startswith("#"))
for k in ("RETRIEVER", "CBT_ENABLED", "GEN_MODEL", "EXT_MODEL", "JUDGE_MODEL", "GUARDRAIL_L2_ENABLED", "MAX_RETRIES"):
    print(f"{k} = {env.get(k, '(mặc định)')}")

In [ ]:
# 8. Seed toàn bộ CSDL (PostgreSQL + pgvector, Neo4j) rồi kiểm số dòng so với file dữ liệu
if os.path.exists(f"{W}/seed.log"):
    os.remove(f"{W}/seed.log")
sh("python ingestion/seed_all.py", log=f"{W}/seed.log")
if "[LỖI" in open(f"{W}/seed.log", encoding="utf-8").read():  # seed_all.py in lỗi nhưng vẫn chạy tiếp
    raise RuntimeError("Seed lỗi, xem seed.log")
import psycopg2
d = f"{APP}/data/processed"
expect = {"nutrition_support": len(json.load(open(f"{d}/400 data dinh duong.json", encoding="utf-8"))),
          "state_effect_map": len(json.load(open(f"{d}/state_effect_map.json", encoding="utf-8"))["map"]),
          "cbt_catalog": len(json.load(open(f"{d}/cbt_catalog.json", encoding="utf-8"))["items"])}
conn = psycopg2.connect(dbname="mental_nutrition_db", user="admin", password="password123", host="localhost")
cur = conn.cursor()
for t, n in expect.items():
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    got = cur.fetchone()[0]
    print(f"{t}: {got} (file: {n})")
    assert got == n, t
cur.execute("SELECT COUNT(*), COUNT(embedding) FROM psychology_knowledge")
rows, emb = cur.fetchone()
n_psy = len(json.load(open(f"{d}/psychology_dataset_vi.json", encoding="utf-8")))
print("psychology_knowledge: dòng", rows, "có embedding", emb, "(file:", n_psy, ")")
assert rows == emb == n_psy
conn.close()
from neo4j import GraphDatabase
with GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123")) as drv:
    c = drv.execute_query("MATCH (n) RETURN count(n) AS c").records[0]["c"]
print("Neo4j nodes:", c)
assert c > 0

In [ ]:
# 9. Truy xuất qua PostgreSQL phải cho cùng đích tác động với bản đọc file JSON, với mọi tổ hợp state
os.chdir(APP)
sys.path.insert(0, APP)
from itertools import product
from backend.constants import STATES, STATE_LABELS
from retrieval.state_retriever import OfflineStateRetriever, StateRetriever
off, pg = OfflineStateRetriever(), StateRetriever()
n = 0
for vals in product(*[[None] + list(STATE_LABELS[s]) for s in STATES]):
    keys = {f"{s}:{v}" for s, v in zip(STATES, vals) if v}
    a, b = off.target_weights(keys), pg.target_weights(keys)
    assert a.keys() == b.keys() and all(abs(a[k] - b[k]) < 1e-4 for k in a), (keys, a, b)
    n += 1
print(f"OK: {n} tổ hợp state, PostgreSQL == JSON")
print("CBT thử:", pg.cbt("Dạo này mình hay lo âu, suy nghĩ tiêu cực liên tục"))

In [ ]:
# 10. Unit test (không cần model)
sh("python -m unittest discover -s tests", log=f"{W}/unittest.log")

In [ ]:
# 11. Chạy thử toàn hệ thống với câu tự soạn (không lấy từ bộ test)
from backend import config
from backend.pipeline import build_default_pipeline
assert config.RETRIEVER == "postgres" and config.JUDGE_MODEL == "gemma3:4b", (config.RETRIEVER, config.JUDGE_MODEL)
p = build_default_pipeline()
for msg in ["Dạo này mình hơi mệt.",
            "Mình đang uống thuốc ngủ bác sĩ kê, có nên ăn gì thêm không?",
            "Tim mình đập loạn, thở không nổi, đang ngồi bệt ngoài hành lang."]:
    t = time.time()
    r = p.handle(f"smoke-{time.time()}", msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:200]}\n")
sid = f"smoke-multi-{time.time()}"
for msg in ["Mình căng thẳng vì công việc, tối nào cũng mất ngủ.", "Mình ăn ít hẳn đi, tâm trạng thì buồn chán."]:
    t = time.time()
    r = p.handle(sid, msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:300]}\n")

## Chạy bộ test
- **Crisis** (164 ca thuộc định nghĩa của nhóm: tự sát, tự hại, hoảng loạn cấp) và **Retrieval** (300 ca): LLM #1 và
  truy xuất qua PostgreSQL của hệ thống đề xuất.
- **P, B1, B2, B3** trên crisis (164) và safety (300): pha 1 sinh câu trả lời, pha 2 chấm bằng giám khảo `gemma3:4b`.
  Dừng giữa chừng thì gắn Output làm input và chạy lại notebook: ca đã xong được bỏ qua.

In [ ]:
if not glob.glob(f"{APP}/logs/eval/crisis_*.jsonl"):
    sh("python -u scripts/eval_system.py crisis", log=f"{APP}/logs/eval/run_crisis_kaggle.txt")

In [ ]:
if not glob.glob(f"{APP}/logs/eval/retrieval_*.jsonl"):
    sh("python -u scripts/eval_system.py retrieval", log=f"{APP}/logs/eval/run_retrieval_kaggle.txt")

In [ ]:
# Hệ thống đề xuất trước, rồi 2 baseline trả lời trực tiếp, cuối cùng pipeline + model gốc (chậm nhất)
for s in ["P", "B1", "B2", "B3"]:
    sh(f"python -u scripts/eval_baselines.py gen --systems {s} --suites crisis safety --workers 2",
       log=f"{APP}/logs/eval/run_gen_{s}.txt")

In [ ]:
sh("python -u scripts/eval_baselines.py score --systems P B1 B2 B3 --suites crisis safety",
   log=f"{APP}/logs/eval/run_score.txt")
sh("python -u scripts/eval_baselines.py report", log=f"{APP}/logs/eval/run_report_kaggle.txt")
# Phân tích đầy đủ như báo cáo: trả lời hợp lệ, đạt hiệu quả, theo loại, phản hồi khẩn cấp, hành vi, McNemar
try:  # lỗi ở bước phân tích không được chặn bước nén kết quả phía sau (phân tích lại được ở máy)
    sh(f"python -u scripts/analyze_final.py {APP}", log=f"{APP}/logs/eval/run_analyze_final.txt")
except RuntimeError as e:
    print("BƯỚC PHÂN TÍCH LỖI, vẫn nén kết quả:", e)

In [ ]:
# Nén kết quả để tải về (logs/eval: jsonl từng ca + bảng tổng hợp; logs/turns: trace từng lượt của pipeline)
sh("ollama list > logs/eval/ollama_list_kaggle.txt && nvidia-smi > logs/eval/nvidia_smi_kaggle.txt")
shutil.make_archive(f"{W}/eval_final_results", "zip", APP, "logs")
print(f"{W}/eval_final_results.zip", round(os.path.getsize(f"{W}/eval_final_results.zip") / 1e6, 1), "MB")